# Coastal Water Quality, Notebook 05: AI Water-Quality Retrieval (MDN)

> Terms are defined in [`GLOSSARY.md`](./GLOSSARY.md). This notebook complements the
> physics-based turbidity estimate in [`04_quantitative_turbidity.ipynb`](./04_quantitative_turbidity.ipynb).

**Scene:** `20250302_030003_92_4001`, Borneo / Makassar Strait coast.

## Purpose

Notebook 04 estimates turbidity with a published physical model. This notebook uses the
Mixture Density Network (MDN) from Pahlevan, Smith and colleagues to estimate chlorophyll-a,
total suspended solids (TSS), and CDOM. The model was trained with GLORIA water samples and
produces real-unit estimates without local field data.

## Processing

The MDN requires TensorFlow 2.5 and runs in the isolated `tanager-mdn` environment through
`scripts/run_mdn_retrieval.py`. That script resamples Tanager to Sentinel-3 OLCI bands, builds
remote-sensing reflectance with SWIR de-glinting, runs the MDN, and saves
`mdn_olci_products.npz`. This notebook loads and interprets those saved maps.

## Caveats

1. The input is land-corrected reflectance, resampled and de-glinted, not water-corrected reflectance.
2. GLORIA represents other waters, not this Borneo coast.
3. The products are quasi-absolute and not field-validated. Use patterns and order of magnitude
   more confidently than exact values.

## 0. Load the AI products

The notebook loads the three MDN maps and water mask. It also rebuilds the Notebook 04
Dogliotti FNU estimate for a direct sediment-pattern comparison with AI TSS.


In [ ]:
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import h5py, rasterio
from scipy.stats import spearmanr

def find_root(start=Path.cwd()):
    p = start.resolve()
    for c in [p, *p.parents]:
        if (c / "data" / "inventory").exists():
            return c
    return p

PROJECT_ROOT = find_root()
SCENE_ID = "20250302_030003_92_4001"
DATA_DIR = PROJECT_ROOT / "data" / "coastal" / SCENE_ID
FIG_DIR  = PROJECT_ROOT / "figures"; FIG_DIR.mkdir(parents=True, exist_ok=True)
NPZ = DATA_DIR / "mdn_olci_products.npz"
SR_PATH, RGB_PATH = DATA_DIR / "ortho_sr_hdf5.h5", DATA_DIR / "ortho_visual.tif"
DF, FILL = "HDFEOS/GRIDS/HYP/Data Fields", -9999.0
assert NPZ.exists(), f"Missing {NPZ} - run (in the tanager-mdn env): python scripts/run_mdn_retrieval.py"

d = np.load(NPZ)
water = d["water"]
chl, tss, cdom = d["chl"], d["tss"], d["cdom"]      # mg/m3, g/m3, m^-1
def wonly(a): return np.where(water & np.isfinite(a), a, np.nan)

# Rebuild Notebook 04 FNU with the same Dogliotti method.
A_RED, C_RED, A_NIR, C_NIR, B_LO, B_HI = 228.1, 0.1641, 3078.9, 0.2112, 0.05, 0.07
with h5py.File(SR_PATH, "r") as f:
    SR = f[f"{DF}/surface_reflectance"]; wl = np.asarray(SR.attrs["wavelengths"], float)
    gw = np.asarray(SR.attrs["good_wavelengths"]).astype(bool)
    def winmean(c, half=6.0):
        idx = np.where((wl >= c-half) & (wl <= c+half) & gw)[0]
        a = SR[idx].astype("float32"); a[a == FILL] = np.nan; return np.nanmean(a, axis=0)
    def band(nm):
        i = int(np.argmin(np.abs(wl - nm))); b = SR[i].astype("float32"); b[b == FILL] = np.nan; return b
    red645, nir859, swir = winmean(645), winmean(859), band(1610)
red_c, nir_c = np.clip(red645 - swir, 0, None), np.clip(nir859 - swir, 0, None)
def _single(rho, A, C): r = np.clip(rho, 0, 0.9 * C); return A * r / (1 - r / C)
w_nir = np.clip((red_c - B_LO) / (B_HI - B_LO), 0, 1)
sat = (red_c > 0.9 * C_RED) | (nir_c > 0.9 * C_NIR)
turbidity = np.where(water & ~sat, (1 - w_nir) * _single(red_c, A_RED, C_RED) + w_nir * _single(nir_c, A_NIR, C_NIR), np.nan)  # FNU

with rasterio.open(RGB_PATH) as ds:
    rgb = np.dstack([ds.read(i) for i in (1, 2, 3)]).astype("float32")
rgb_disp = np.clip(rgb / np.percentile(rgb[rgb > 0], 98), 0, 1)

print(f"water pixels: {int(water.sum()):,}")
for k, a, u in [("turbidity", turbidity, "FNU"), ("chl", chl, "mg/m3"), ("TSS", tss, "g/m3"), ("CDOM a440", cdom, "1/m")]:
    v = a[water & np.isfinite(a)]
    print(f"  {k:9s}: p5/median/p95 = {np.percentile(v,5):.2f} / {np.median(v):.2f} / {np.percentile(v,95):.2f} {u}")



## 1. The MDN

A Mixture Density Network predicts a probability distribution, so it returns an estimate and
its uncertainty. Pahlevan, Smith and colleagues trained the model with GLORIA, a global library
of water samples with measured reflectance, chlorophyll, sediment, and CDOM.

This analysis uses the Sentinel-3 OLCI model with 12 visible and near-infrared bands. It returns:

- **Chlorophyll-a** (mg·m⁻³): phytoplankton or algae amount.
- **TSS** (g·m⁻³, equivalent to mg·L⁻¹): suspended sediment load.
- **CDOM** as **a_cdom(440)** (m⁻¹): coloured dissolved-matter absorption.

**References used here**

- Pahlevan, N., et al. (2020). *Seamless retrievals of chlorophyll-a from Sentinel-2 (MSI) and Sentinel-3 (OLCI) in
  inland and coastal waters: A machine-learning approach.* Remote Sensing of Environment, 240, 111604.
  https://doi.org/10.1016/j.rse.2019.111604. This is the MDN model for OLCI used here.
- Pahlevan, N., et al. (2022). *Simultaneous retrieval of selected optical water quality indicators from Landsat-8,
  Sentinel-2, and Sentinel-3.* Remote Sensing of Environment, 270, 112860.
  https://doi.org/10.1016/j.rse.2021.112860. This is the combined Chla, TSS, and a_cdom(440) MDN.
- Pahlevan, N., et al. (2021). *Hyperspectral retrievals of phytoplankton absorption and chlorophyll-a in inland and
  nearshore coastal waters.* Remote Sensing of Environment, 253, 112200.
  https://doi.org/10.1016/j.rse.2020.112200. This supports the MDN and GLORIA basis for hyperspectral sensors.

## 2. AI maps

TSS should identify the river-mouth and coastal sediment plume, consistent with Notebook 04
turbidity. Chlorophyll should remain low, as in Notebooks 02 and 03. CDOM may track sediment,
a known Case-2 confound.


In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(17, 6))
panels = [("Chlorophyll-a (mg/m3)", wonly(chl), "viridis"),
          ("TSS - suspended sediment (g/m3)", wonly(tss), "turbo"),
          ("CDOM  a_cdom(440)  (1/m)", wonly(cdom), "magma")]
for a, (ttl, img, cm) in zip(ax, panels):
    im = a.imshow(img, cmap=cm, vmin=np.nanpercentile(img, 2), vmax=np.nanpercentile(img, 98))
    a.set_title(ttl); a.axis("off"); fig.colorbar(im, ax=a, fraction=0.046)
fig.tight_layout(); fig.savefig(FIG_DIR / "05_mdn_maps.png", dpi=150, bbox_inches="tight")
plt.show(); print("saved ->", FIG_DIR / "05_mdn_maps.png")



## 3. Physics and AI comparison

Notebook 04 turbidity is in FNU and Notebook 05 TSS is in g·m⁻³. They measure related but
different properties. Rank correlation tests whether the maps recover the same sediment pattern
without a conversion factor.

The factor `k = typical TSS / typical turbidity` relates the two medians. It is not evidence of
agreement and not a calibrated turbidity-to-sediment relationship. The comparison checks whether
the same factor also describes the distribution tails. Chlorophyll and CDOM are reported only by
this notebook; their values are checked against the qualitative findings from Notebooks 02 and 03.


In [ ]:
m = water & np.isfinite(turbidity) & np.isfinite(tss) & np.isfinite(chl) & np.isfinite(cdom)
r_tss_turb = spearmanr(turbidity[m], tss[m]).statistic
r_tss_cdom = spearmanr(tss[m], cdom[m]).statistic

def mr(a):
    v = a[m]; return np.median(v), np.percentile(v, 5), np.percentile(v, 95)
tm, tlo, thi = mr(turbidity)
sm, slo, shi = mr(tss)
cm, clo, chi = mr(chl)
dm, dlo, dhi = mr(cdom)
# Use medians to avoid a skew-dominated regression.
k = sm / tm                       # g/m3 of TSS per FNU of turbidity

print("WATER-QUALITY NUMBERS   (median over water; range = 5th-95th percentile)\n")
print("  SEDIMENT")
print(f"    turbidity (nb4 physics) : {tm:5.1f} FNU    (range {tlo:.1f} - {thi:.0f})")
print(f"    TSS       (nb5 AI)      : {sm:5.1f} g/m3   (range {slo:.1f} - {shi:.0f})")
print(f"  CHLOROPHYLL-a (nb5 AI)    : {cm:5.2f} mg/m3  (range {clo:.1f} - {chi:.0f})   low: only {100*np.mean(chl[m]>10):.0f}% above 10 (no bloom)")
print(f"  CDOM a(440)   (nb5 AI)    : {dm:5.2f} 1/m    (range {dlo:.2f} - {dhi:.2f})  tracks sediment (Spearman {r_tss_cdom:+.2f})")
print()
print("  Turbidity (FNU) and TSS (g/m3) are different units for the SAME suspended sediment.")
print("  We ask two SEPARATE questions, and keep them separate:\n")
print(f"  1) DO THEY AGREE?  Rank correlation between the two maps = Spearman {r_tss_turb:+.2f}.")
print( "     This uses the pixel values directly - it needs NO conversion factor, so it is the real, independent")
print( "     evidence that physics (notebook 4) and the AI (this notebook) see the same sediment pattern.")
print(f"  2) HOW DO THEIR SCALES RELATE?  Define a unit-bridge k = (typical TSS)/(typical turbidity) = "
      f"{sm:.1f}/{tm:.1f} = {k:.2f} g/m3 per FNU.")
print( "     This matches the two medians BY DEFINITION (trivial - do not read it as 'agreement'). The meaningful")
print( "     test is whether the SAME k - which was NOT fitted to the tails - also fits the tails:")
print(f"         5th pct  : nb5 TSS {slo:4.1f}   vs   k x nb4 turbidity = {k*tlo:4.1f} g/m3")
print(f"        95th pct  : nb5 TSS {shi:4.0f}    vs   k x nb4 turbidity = {k*thi:4.0f} g/m3")
print( "     One constant spanning low/mid/high => the two products are proportionally scaled copies (a real result).")
print()
print(f"  CAVEAT: k = {k:.2f} is only the RATIO OF OUR OWN TWO ESTIMATES (nb5 TSS / nb4 turbidity) - a rough unit-bridge,")
print( "  NOT a calibrated turbidity->sediment law (that needs in-situ data we don't have). Use it as an")
print(f"  order-of-magnitude guide only: TSS (g/m3) ~ {k:.2f} x turbidity, e.g. 20 FNU ~ {k*20:.0f} g/m3, 50 FNU ~ {k*50:.0f}.")

fig, ax = plt.subplots(1, 3, figsize=(18, 6))
im0 = ax[0].imshow(np.where(water, turbidity, np.nan), cmap="turbo", vmin=0, vmax=np.nanpercentile(turbidity[m], 98))
ax[0].set_title("Notebook 4 physics - turbidity (FNU)"); ax[0].axis("off"); fig.colorbar(im0, ax=ax[0], fraction=0.046, label="FNU")
im1 = ax[1].imshow(wonly(tss), cmap="turbo", vmin=0, vmax=np.nanpercentile(tss[m], 98))
ax[1].set_title("Notebook 5 AI - TSS (g/m3)"); ax[1].axis("off"); fig.colorbar(im1, ax=ax[1], fraction=0.046, label="g/m3")

rng = np.random.default_rng(0); idx = rng.choice(int(m.sum()), size=min(6000, int(m.sum())), replace=False)
ax[2].scatter(turbidity[m][idx], tss[m][idx], s=4, alpha=0.3)
xs = np.linspace(0, np.percentile(turbidity[m], 99), 50)
ax[2].plot(xs, k * xs, "r-", lw=2, label=f"unit-bridge: TSS ~ {k:.2f} x FNU")
ax[2].set_xlabel("Notebook 4 turbidity (FNU)"); ax[2].set_ylabel("Notebook 5 TSS (g/m3)")
ax[2].set_title(f"same sediment pattern, two methods  (Spearman {r_tss_turb:+.2f})"); ax[2].legend()
fig.tight_layout(); fig.savefig(FIG_DIR / "05_sediment_compare.png", dpi=150, bbox_inches="tight")
plt.show(); print("saved ->", FIG_DIR / "05_sediment_compare.png")


In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(14, 5))
ax[0].hist(chl[m][chl[m] < np.percentile(chl[m], 99)], bins=80, color="seagreen")
ax[0].axvline(np.median(chl[m]), color="k", ls="--", label=f"median {np.median(chl[m]):.1f} mg/m3")
ax[0].set_xlabel("AI chlorophyll-a (mg/m3)"); ax[0].set_ylabel("water pixels")
ax[0].set_title("Chlorophyll - low, consistent with 'no bloom'"); ax[0].legend()
ax[1].scatter(tss[m][idx], cdom[m][idx], s=4, alpha=0.3, color="C4")
ax[1].set_xlabel("AI TSS (g/m3)"); ax[1].set_ylabel("AI CDOM a_cdom(440) (1/m)")
ax[1].set_title(f"CDOM re-reads sediment  (Spearman {r_tss_cdom:+.2f})")
fig.savefig(FIG_DIR / "05_crosscheck.png", dpi=150, bbox_inches="tight")
plt.show(); print("saved ->", FIG_DIR / "05_crosscheck.png")



## Summary

Notebook 05 adds independent MDN maps of chlorophyll-a, TSS, and CDOM. The TSS pattern provides
a separate check on the sediment plume mapped in Notebook 04. Chlorophyll remains low. CDOM is
strongly linked to sediment and is the least reliable product.

Exact values remain uncertain because the training is global, the input reflectance is land-based,
and there is no local validation. The main value is agreement between independent physics and AI
sediment estimates.

**Figures:** `05_mdn_maps.png`, `05_sediment_compare.png`, `05_crosscheck.png`.

Further improvement requires water-specific atmospheric correction and near-coincident
Sentinel-2 or Sentinel-3 comparison.
